# Lesson 10b: Continuous Control Practical

**Learning Objectives:**
- Implement DDPG (Deep Deterministic Policy Gradient) from scratch
- Implement TD3 (Twin Delayed DDPG) with all improvements
- Implement SAC (Soft Actor-Critic) with automatic temperature tuning
- Build efficient replay buffer and exploration mechanisms
- Train agents on continuous control tasks
- Compare performance of DDPG, TD3, and SAC

**Prerequisites:** Policy Gradients (8b), PPO (9b), Continuous Control Theory (10a)

**Environment:** Google Colab with GPU support recommended

In [ ]:
# Install dependencies
!pip install gymnasium[mujoco] numpy matplotlib torch tqdm -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque
import random
import gymnasium as gym
from tqdm import tqdm
import copy

# Set random seeds
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Replay Buffer

In [ ]:
class ReplayBuffer:
    """Experience replay buffer for off-policy learning."""
    
    def __init__(self, capacity: int = 1000000):
        """
        Args:
            capacity: Maximum number of transitions to store
        """
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """Add transition to buffer."""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size: int):
        """Sample random batch of transitions."""
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        return (
            np.array(states),
            np.array(actions),
            np.array(rewards),
            np.array(next_states),
            np.array(dones)
        )
    
    def __len__(self):
        return len(self.buffer)

# Test replay buffer
buffer = ReplayBuffer(capacity=100)
for i in range(10):
    buffer.push(
        state=np.random.randn(4),
        action=np.random.randn(2),
        reward=np.random.randn(),
        next_state=np.random.randn(4),
        done=False
    )

states, actions, rewards, next_states, dones = buffer.sample(5)
print(f"Buffer size: {len(buffer)}")
print(f"Sampled states shape: {states.shape}")
print(f"Sampled actions shape: {actions.shape}")

## 2. Neural Network Architectures

In [ ]:
class Actor(nn.Module):
    """Deterministic policy network for DDPG/TD3."""
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 hidden_sizes: list = [400, 300]):
        super(Actor, self).__init__()
        self.max_action = max_action
        
        # Build network
        layers = []
        input_dim = state_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_dim, hidden_size))
            layers.append(nn.ReLU())
            input_dim = hidden_size
        
        layers.append(nn.Linear(input_dim, action_dim))
        layers.append(nn.Tanh())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, state):
        """Forward pass: state -> action."""
        return self.max_action * self.network(state)


class Critic(nn.Module):
    """Q-function network for DDPG/TD3."""
    
    def __init__(self, state_dim: int, action_dim: int,
                 hidden_sizes: list = [400, 300]):
        super(Critic, self).__init__()
        
        # Build network: concatenate state and action
        layers = []
        input_dim = state_dim + action_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_dim, hidden_size))
            layers.append(nn.ReLU())
            input_dim = hidden_size
        
        layers.append(nn.Linear(input_dim, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, state, action):
        """Forward pass: (state, action) -> Q-value."""
        x = torch.cat([state, action], dim=-1)
        return self.network(x)


# Test networks
state_dim, action_dim, max_action = 4, 2, 1.0
actor = Actor(state_dim, action_dim, max_action).to(device)
critic = Critic(state_dim, action_dim).to(device)

test_state = torch.randn(1, state_dim).to(device)
test_action = actor(test_state)
test_q = critic(test_state, test_action)

print(f"Actor output shape: {test_action.shape}")
print(f"Action range: [{test_action.min().item():.3f}, {test_action.max().item():.3f}]")
print(f"Critic output: {test_q.item():.3f}")

## 3. DDPG Implementation

In [ ]:
class DDPGAgent:
    """Deep Deterministic Policy Gradient agent."""
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 lr_actor: float = 1e-3, lr_critic: float = 1e-3,
                 gamma: float = 0.99, tau: float = 0.005,
                 buffer_size: int = 1000000, batch_size: int = 256):
        """
        Args:
            state_dim: Dimension of state space
            action_dim: Dimension of action space
            max_action: Maximum action value
            lr_actor: Actor learning rate
            lr_critic: Critic learning rate
            gamma: Discount factor
            tau: Target network soft update rate
            buffer_size: Replay buffer capacity
            batch_size: Minibatch size
        """
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.max_action = max_action
        
        # Actor networks
        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        # Critic networks
        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(buffer_size)
    
    def select_action(self, state, noise_scale: float = 0.1):
        """Select action with Gaussian exploration noise."""
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.actor(state).cpu().numpy()[0]
        
        # Add exploration noise
        if noise_scale > 0:
            noise = np.random.normal(0, noise_scale, size=action.shape)
            action = action + noise
            action = np.clip(action, -self.max_action, self.max_action)
        
        return action
    
    def update(self):
        """Perform one update step."""
        if len(self.replay_buffer) < self.batch_size:
            return {}
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.FloatTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(device)
        
        # =================== Update Critic ===================
        with torch.no_grad():
            # Compute target Q-value
            next_actions = self.actor_target(next_states)
            target_q = self.critic_target(next_states, next_actions)
            target_q = rewards + (1 - dones) * self.gamma * target_q
        
        # Current Q-value
        current_q = self.critic(states, actions)
        
        # Critic loss
        critic_loss = F.mse_loss(current_q, target_q)
        
        # Update critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        # =================== Update Actor ===================
        # Actor loss: maximize Q(s, μ(s))
        actor_loss = -self.critic(states, self.actor(states)).mean()
        
        # Update actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # =================== Update Target Networks ===================
        self._soft_update(self.actor, self.actor_target)
        self._soft_update(self.critic, self.critic_target)
        
        return {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item(),
            'q_value': current_q.mean().item()
        }
    
    def _soft_update(self, source, target):
        """Soft update: target = τ * source + (1 - τ) * target."""
        for target_param, source_param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(
                self.tau * source_param.data + (1 - self.tau) * target_param.data
            )

print("DDPG agent created successfully")

## 4. TD3 Implementation

In [ ]:
class TD3Agent:
    """Twin Delayed Deep Deterministic Policy Gradient agent."""
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 lr_actor: float = 1e-3, lr_critic: float = 1e-3,
                 gamma: float = 0.99, tau: float = 0.005,
                 policy_noise: float = 0.2, noise_clip: float = 0.5,
                 policy_delay: int = 2,
                 buffer_size: int = 1000000, batch_size: int = 256):
        """
        Args:
            policy_noise: Std of noise for target policy smoothing
            noise_clip: Range to clip noise
            policy_delay: Delay actor updates (every d critic updates)
        """
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.max_action = max_action
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_delay = policy_delay
        self.update_counter = 0
        
        # Actor networks
        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        # Twin critic networks
        self.critic1 = Critic(state_dim, action_dim).to(device)
        self.critic1_target = copy.deepcopy(self.critic1)
        self.critic1_optimizer = optim.Adam(self.critic1.parameters(), lr=lr_critic)
        
        self.critic2 = Critic(state_dim, action_dim).to(device)
        self.critic2_target = copy.deepcopy(self.critic2)
        self.critic2_optimizer = optim.Adam(self.critic2.parameters(), lr=lr_critic)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(buffer_size)
    
    def select_action(self, state, noise_scale: float = 0.1):
        """Select action with exploration noise."""
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.actor(state).cpu().numpy()[0]
        
        if noise_scale > 0:
            noise = np.random.normal(0, noise_scale, size=action.shape)
            action = action + noise
            action = np.clip(action, -self.max_action, self.max_action)
        
        return action
    
    def update(self):
        """Perform one update step with TD3 improvements."""
        if len(self.replay_buffer) < self.batch_size:
            return {}
        
        self.update_counter += 1
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.FloatTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(device)
        
        # =================== Update Critics ===================
        with torch.no_grad():
            # Target policy smoothing: add clipped noise
            noise = torch.randn_like(actions) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            next_actions = self.actor_target(next_states) + noise
            next_actions = next_actions.clamp(-self.max_action, self.max_action)
            
            # Clipped double Q-learning: min of two critics
            target_q1 = self.critic1_target(next_states, next_actions)
            target_q2 = self.critic2_target(next_states, next_actions)
            target_q = torch.min(target_q1, target_q2)
            target_q = rewards + (1 - dones) * self.gamma * target_q
        
        # Current Q-values
        current_q1 = self.critic1(states, actions)
        current_q2 = self.critic2(states, actions)
        
        # Critic losses
        critic1_loss = F.mse_loss(current_q1, target_q)
        critic2_loss = F.mse_loss(current_q2, target_q)
        
        # Update critic 1
        self.critic1_optimizer.zero_grad()
        critic1_loss.backward()
        self.critic1_optimizer.step()
        
        # Update critic 2
        self.critic2_optimizer.zero_grad()
        critic2_loss.backward()
        self.critic2_optimizer.step()
        
        stats = {
            'critic1_loss': critic1_loss.item(),
            'critic2_loss': critic2_loss.item(),
            'q_value': current_q1.mean().item()
        }
        
        # =================== Delayed Policy Update ===================
        if self.update_counter % self.policy_delay == 0:
            # Actor loss: maximize Q1(s, μ(s))
            actor_loss = -self.critic1(states, self.actor(states)).mean()
            
            # Update actor
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            # Update target networks
            self._soft_update(self.actor, self.actor_target)
            self._soft_update(self.critic1, self.critic1_target)
            self._soft_update(self.critic2, self.critic2_target)
            
            stats['actor_loss'] = actor_loss.item()
        
        return stats
    
    def _soft_update(self, source, target):
        """Soft update target network."""
        for target_param, source_param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(
                self.tau * source_param.data + (1 - self.tau) * target_param.data
            )

print("TD3 agent created successfully")

## 5. SAC Networks (Stochastic Policy)

In [ ]:
class GaussianPolicy(nn.Module):
    """Stochastic Gaussian policy for SAC."""
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 hidden_sizes: list = [400, 300],
                 log_std_min: float = -20, log_std_max: float = 2):
        super(GaussianPolicy, self).__init__()
        self.max_action = max_action
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max
        
        # Shared layers
        layers = []
        input_dim = state_dim
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_dim, hidden_size))
            layers.append(nn.ReLU())
            input_dim = hidden_size
        self.shared = nn.Sequential(*layers)
        
        # Output heads
        self.mean = nn.Linear(input_dim, action_dim)
        self.log_std = nn.Linear(input_dim, action_dim)
    
    def forward(self, state):
        """Forward pass: state -> (mean, log_std)."""
        x = self.shared(state)
        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std
    
    def sample(self, state):
        """Sample action using reparameterization trick."""
        mean, log_std = self.forward(state)
        std = log_std.exp()
        
        # Reparameterization trick
        normal = torch.distributions.Normal(mean, std)
        x_t = normal.rsample()  # Reparameterized sample
        
        # Apply tanh squashing
        action = torch.tanh(x_t) * self.max_action
        
        # Compute log probability with squashing correction
        log_prob = normal.log_prob(x_t)
        # Correction for tanh squashing
        log_prob -= torch.log(self.max_action * (1 - torch.tanh(x_t).pow(2)) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        
        return action, log_prob
    
    def evaluate(self, state):
        """Get deterministic action (for evaluation)."""
        mean, _ = self.forward(state)
        return torch.tanh(mean) * self.max_action


# Test Gaussian policy
policy = GaussianPolicy(state_dim=4, action_dim=2, max_action=1.0).to(device)
test_state = torch.randn(5, 4).to(device)
test_action, test_log_prob = policy.sample(test_state)
print(f"Action shape: {test_action.shape}")
print(f"Log prob shape: {test_log_prob.shape}")
print(f"Action range: [{test_action.min().item():.3f}, {test_action.max().item():.3f}]")

## 6. SAC Implementation

In [ ]:
class SACAgent:
    """Soft Actor-Critic agent with automatic temperature tuning."""
    
    def __init__(self, state_dim: int, action_dim: int, max_action: float,
                 lr_actor: float = 3e-4, lr_critic: float = 3e-4, lr_alpha: float = 3e-4,
                 gamma: float = 0.99, tau: float = 0.005,
                 alpha: float = 0.2, auto_tune_alpha: bool = True,
                 buffer_size: int = 1000000, batch_size: int = 256):
        """
        Args:
            alpha: Initial temperature parameter
            auto_tune_alpha: Whether to automatically tune temperature
            lr_alpha: Learning rate for temperature parameter
        """
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.max_action = max_action
        self.auto_tune_alpha = auto_tune_alpha
        
        # Actor network
        self.actor = GaussianPolicy(state_dim, action_dim, max_action).to(device)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        
        # Twin critic networks
        self.critic1 = Critic(state_dim, action_dim).to(device)
        self.critic1_target = copy.deepcopy(self.critic1)
        self.critic1_optimizer = optim.Adam(self.critic1.parameters(), lr=lr_critic)
        
        self.critic2 = Critic(state_dim, action_dim).to(device)
        self.critic2_target = copy.deepcopy(self.critic2)
        self.critic2_optimizer = optim.Adam(self.critic2.parameters(), lr=lr_critic)
        
        # Temperature parameter
        if self.auto_tune_alpha:
            # Target entropy = -dim(A)
            self.target_entropy = -action_dim
            self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
            self.alpha = self.log_alpha.exp()
            self.alpha_optimizer = optim.Adam([self.log_alpha], lr=lr_alpha)
        else:
            self.alpha = alpha
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(buffer_size)
    
    def select_action(self, state, evaluate: bool = False):
        """Select action (stochastic during training, deterministic for eval)."""
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            if evaluate:
                action = self.actor.evaluate(state)
            else:
                action, _ = self.actor.sample(state)
        return action.cpu().numpy()[0]
    
    def update(self):
        """Perform one SAC update step."""
        if len(self.replay_buffer) < self.batch_size:
            return {}
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.FloatTensor(actions).to(device)
        rewards = torch.FloatTensor(rewards).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).unsqueeze(1).to(device)
        
        # =================== Update Critics ===================
        with torch.no_grad():
            # Sample next actions
            next_actions, next_log_probs = self.actor.sample(next_states)
            
            # Clipped double Q-learning
            target_q1 = self.critic1_target(next_states, next_actions)
            target_q2 = self.critic2_target(next_states, next_actions)
            target_q = torch.min(target_q1, target_q2)
            
            # Entropy-augmented target
            if self.auto_tune_alpha:
                alpha = self.log_alpha.exp()
            else:
                alpha = self.alpha
            target_q = target_q - alpha * next_log_probs
            target_q = rewards + (1 - dones) * self.gamma * target_q
        
        # Current Q-values
        current_q1 = self.critic1(states, actions)
        current_q2 = self.critic2(states, actions)
        
        # Critic losses
        critic1_loss = F.mse_loss(current_q1, target_q)
        critic2_loss = F.mse_loss(current_q2, target_q)
        
        # Update critics
        self.critic1_optimizer.zero_grad()
        critic1_loss.backward()
        self.critic1_optimizer.step()
        
        self.critic2_optimizer.zero_grad()
        critic2_loss.backward()
        self.critic2_optimizer.step()
        
        # =================== Update Actor ===================
        # Sample actions from current policy
        new_actions, log_probs = self.actor.sample(states)
        
        # Q-values for new actions
        q1_new = self.critic1(states, new_actions)
        q2_new = self.critic2(states, new_actions)
        q_new = torch.min(q1_new, q2_new)
        
        # Actor loss: maximize (Q - α*log_prob)
        if self.auto_tune_alpha:
            alpha = self.log_alpha.exp().detach()
        else:
            alpha = self.alpha
        actor_loss = (alpha * log_probs - q_new).mean()
        
        # Update actor
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # =================== Update Temperature ===================
        alpha_loss = None
        if self.auto_tune_alpha:
            # α loss: minimize E[-α * log π(a|s) - α * H_target]
            alpha_loss = -(self.log_alpha.exp() * (log_probs + self.target_entropy).detach()).mean()
            
            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()
            
            self.alpha = self.log_alpha.exp()
        
        # =================== Update Target Networks ===================
        self._soft_update(self.critic1, self.critic1_target)
        self._soft_update(self.critic2, self.critic2_target)
        
        stats = {
            'critic1_loss': critic1_loss.item(),
            'critic2_loss': critic2_loss.item(),
            'actor_loss': actor_loss.item(),
            'q_value': current_q1.mean().item(),
            'entropy': -log_probs.mean().item(),
            'alpha': alpha if not self.auto_tune_alpha else self.alpha.item()
        }
        
        if alpha_loss is not None:
            stats['alpha_loss'] = alpha_loss.item()
        
        return stats
    
    def _soft_update(self, source, target):
        """Soft update target network."""
        for target_param, source_param in zip(target.parameters(), source.parameters()):
            target_param.data.copy_(
                self.tau * source_param.data + (1 - self.tau) * target_param.data
            )

print("SAC agent created successfully")

## 7. Training Function

In [ ]:
def train_agent(agent, env_name: str, num_episodes: int = 100,
                max_steps: int = 1000, warmup_steps: int = 1000,
                eval_freq: int = 10, noise_scale: float = 0.1):
    """Train continuous control agent.
    
    Args:
        agent: DDPG, TD3, or SAC agent
        env_name: Gymnasium environment name
        num_episodes: Number of training episodes
        max_steps: Maximum steps per episode
        warmup_steps: Random exploration steps before training
        eval_freq: Evaluation frequency (episodes)
        noise_scale: Exploration noise scale
    """
    env = gym.make(env_name)
    eval_env = gym.make(env_name)
    
    episode_rewards = []
    eval_rewards = []
    total_steps = 0
    
    print(f"Training {agent.__class__.__name__} on {env_name}...")
    
    for episode in tqdm(range(num_episodes)):
        state, _ = env.reset(seed=SEED + episode)
        episode_reward = 0
        
        for step in range(max_steps):
            total_steps += 1
            
            # Select action
            if total_steps < warmup_steps:
                # Random exploration
                action = env.action_space.sample()
            else:
                # Agent policy with noise
                if isinstance(agent, SACAgent):
                    action = agent.select_action(state, evaluate=False)
                else:
                    action = agent.select_action(state, noise_scale=noise_scale)
            
            # Execute action
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            agent.replay_buffer.push(state, action, reward, next_state, done)
            
            # Update agent
            if total_steps >= warmup_steps:
                agent.update()
            
            episode_reward += reward
            state = next_state
            
            if done:
                break
        
        episode_rewards.append(episode_reward)
        
        # Evaluation
        if (episode + 1) % eval_freq == 0:
            eval_reward = evaluate_agent(agent, eval_env, num_episodes=5)
            eval_rewards.append(eval_reward)
            print(f"Episode {episode+1}/{num_episodes}, "
                  f"Train Reward: {np.mean(episode_rewards[-eval_freq:]):.2f}, "
                  f"Eval Reward: {eval_reward:.2f}")
    
    env.close()
    eval_env.close()
    
    return episode_rewards, eval_rewards


def evaluate_agent(agent, env, num_episodes: int = 5):
    """Evaluate agent without exploration noise."""
    total_reward = 0
    
    for _ in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            if isinstance(agent, SACAgent):
                action = agent.select_action(state, evaluate=True)
            else:
                action = agent.select_action(state, noise_scale=0.0)
            
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        total_reward += episode_reward
    
    return total_reward / num_episodes

print("Training functions defined")

## 8. Experiment: Compare DDPG, TD3, and SAC

In [ ]:
# Environment setup
ENV_NAME = "Pendulum-v1"  # Simple continuous control task
env = gym.make(ENV_NAME)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
max_action = float(env.action_space.high[0])

print(f"Environment: {ENV_NAME}")
print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Max action: {max_action}")

env.close()

In [ ]:
# Create agents
print("Creating agents...")

ddpg_agent = DDPGAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    max_action=max_action,
    buffer_size=100000,
    batch_size=128
)

td3_agent = TD3Agent(
    state_dim=state_dim,
    action_dim=action_dim,
    max_action=max_action,
    buffer_size=100000,
    batch_size=128,
    policy_delay=2
)

sac_agent = SACAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    max_action=max_action,
    buffer_size=100000,
    batch_size=128,
    auto_tune_alpha=True
)

print("All agents created!")

In [ ]:
# Train DDPG
print("\n" + "="*50)
print("Training DDPG")
print("="*50)

ddpg_rewards, ddpg_eval = train_agent(
    ddpg_agent,
    ENV_NAME,
    num_episodes=100,
    warmup_steps=500,
    noise_scale=0.1
)

In [ ]:
# Train TD3
print("\n" + "="*50)
print("Training TD3")
print("="*50)

td3_rewards, td3_eval = train_agent(
    td3_agent,
    ENV_NAME,
    num_episodes=100,
    warmup_steps=500,
    noise_scale=0.1
)

In [ ]:
# Train SAC
print("\n" + "="*50)
print("Training SAC")
print("="*50)

sac_rewards, sac_eval = train_agent(
    sac_agent,
    ENV_NAME,
    num_episodes=100,
    warmup_steps=500
)

## 9. Results Visualization

In [ ]:
def moving_average(data, window=10):
    """Compute moving average for smoothing."""
    weights = np.ones(window) / window
    return np.convolve(data, weights, mode='valid')

# Plot training curves
plt.figure(figsize=(15, 5))

# Training rewards
plt.subplot(1, 2, 1)
plt.plot(moving_average(ddpg_rewards, 10), label='DDPG', alpha=0.8)
plt.plot(moving_average(td3_rewards, 10), label='TD3', alpha=0.8)
plt.plot(moving_average(sac_rewards, 10), label='SAC', alpha=0.8)
plt.xlabel('Episode')
plt.ylabel('Training Reward')
plt.title('Training Performance Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

# Evaluation rewards
plt.subplot(1, 2, 2)
eval_episodes = np.arange(10, 101, 10)
plt.plot(eval_episodes, ddpg_eval, 'o-', label='DDPG', alpha=0.8)
plt.plot(eval_episodes, td3_eval, 's-', label='TD3', alpha=0.8)
plt.plot(eval_episodes, sac_eval, '^-', label='SAC', alpha=0.8)
plt.xlabel('Episode')
plt.ylabel('Evaluation Reward')
plt.title('Evaluation Performance Comparison')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/continuous_control_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFinal Evaluation Results:")
print(f"DDPG: {ddpg_eval[-1]:.2f}")
print(f"TD3:  {td3_eval[-1]:.2f}")
print(f"SAC:  {sac_eval[-1]:.2f}")

## 10. Ablation Study: TD3 Components

In [ ]:
print("Ablation Study: Impact of TD3 improvements\n")

# 1. DDPG (baseline)
print("1. DDPG (baseline)")
print("   - Single critic")
print("   - No delayed updates")
print("   - No target smoothing")
print(f"   Final reward: {ddpg_eval[-1]:.2f}\n")

# 2. TD3 (all improvements)
print("2. TD3 (all improvements)")
print("   - Twin critics (clipped double-Q)")
print("   - Delayed policy updates")
print("   - Target policy smoothing")
print(f"   Final reward: {td3_eval[-1]:.2f}")
print(f"   Improvement: {(td3_eval[-1] - ddpg_eval[-1]):.2f}\n")

# 3. SAC (entropy regularization)
print("3. SAC (maximum entropy)")
print("   - Twin critics")
print("   - Stochastic policy")
print("   - Entropy regularization")
print("   - Auto-tuned temperature")
print(f"   Final reward: {sac_eval[-1]:.2f}")
print(f"   Improvement over DDPG: {(sac_eval[-1] - ddpg_eval[-1]):.2f}")

## 11. Visualization: Policy Behavior

In [ ]:
def visualize_policy(agent, env_name, num_steps=200):
    """Visualize learned policy behavior."""
    env = gym.make(env_name, render_mode="rgb_array")
    state, _ = env.reset()
    
    states = []
    actions = []
    rewards = []
    
    for _ in range(num_steps):
        if isinstance(agent, SACAgent):
            action = agent.select_action(state, evaluate=True)
        else:
            action = agent.select_action(state, noise_scale=0.0)
        
        states.append(state)
        actions.append(action)
        
        state, reward, terminated, truncated, _ = env.step(action)
        rewards.append(reward)
        
        if terminated or truncated:
            state, _ = env.reset()
    
    env.close()
    
    return np.array(states), np.array(actions), np.array(rewards)

# Visualize SAC policy (best performing)
states, actions, rewards = visualize_policy(sac_agent, ENV_NAME)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(states[:, 0], label='cos(θ)')
plt.plot(states[:, 1], label='sin(θ)')
plt.plot(states[:, 2], label='θ_dot')
plt.xlabel('Step')
plt.ylabel('State')
plt.title('State Trajectory (SAC)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(actions[:, 0])
plt.xlabel('Step')
plt.ylabel('Action (Torque)')
plt.title('Action Trajectory (SAC)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(rewards)
plt.xlabel('Step')
plt.ylabel('Reward')
plt.title('Reward Trajectory (SAC)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/sac_policy_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average reward over trajectory: {rewards.mean():.2f}")

## Summary

### Implementations Completed

1. **Replay Buffer**: Efficient experience storage for off-policy learning

2. **DDPG**:
   - Deterministic actor network
   - Single critic network
   - Target networks with soft updates
   - Gaussian exploration noise

3. **TD3**:
   - Twin critics (clipped double-Q learning)
   - Delayed policy updates
   - Target policy smoothing
   - Improved stability over DDPG

4. **SAC**:
   - Stochastic Gaussian policy
   - Twin critics
   - Entropy regularization
   - Automatic temperature tuning
   - Best overall performance

### Key Insights

1. **Sample efficiency**: All three algorithms are off-policy and sample efficient

2. **Stability**: TD3 > DDPG, SAC > TD3 (entropy regularization helps)

3. **Performance**: SAC typically achieves best results due to automatic exploration

4. **Hyperparameter sensitivity**: SAC is most robust (auto-tuned α)

5. **Use cases**:
   - Robotics: TD3 or SAC
   - Games: SAC
   - Simple tasks: Any works
   - Complex tasks: SAC recommended

### Next Steps

- **Lesson 11**: Model-based RL (learn environment dynamics)
- **Lesson 12**: Advanced exploration strategies
- Try MuJoCo environments (HalfCheetah, Ant, Humanoid)
- Implement prioritized replay buffer
- Add hindsight experience replay (HER) for goal-conditioned tasks

## Exercises

### Implementation Exercises

1. **Modify TD3** to use different policy delay values (1, 2, 5) and compare stability

2. **Implement Ornstein-Uhlenbeck noise** for DDPG exploration instead of Gaussian

3. **Add prioritized experience replay** to any agent and measure improvement

4. **Implement parameter space noise** (Plappert et al. 2017) for exploration

5. **Create a critic that processes state first**, then combines with action (SAC style)

### Analysis Exercises

6. **Plot Q-value evolution** during training for each algorithm

7. **Visualize entropy decay** in SAC training

8. **Compare convergence speed** on different MuJoCo tasks

9. **Ablation study**: Remove each TD3 component individually

10. **Study effect of replay buffer size** (10K, 100K, 1M)

### Application Exercises

11. **Train on Ant-v4** (more complex locomotion) and compare algorithms

12. **Implement goal-conditioned RL** with HER (Hindsight Experience Replay)

13. **Add state/reward normalization** and measure impact

14. **Create ensemble of critics** (3+) and take minimum

15. **Implement distributed SAC** with multiple workers collecting experience